In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import polars as pl
from tqdm.auto import tqdm

from tracking_cellmot.io import open_dataset

ROOT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

DATA_ROOT = ROOT / "data/raw/competition/biohub-cell-tracking-during-development"

TRAIN_DIR = DATA_ROOT / "train"

train_zarrs = sorted(
    TRAIN_DIR.glob("*.zarr")
)

len(train_zarrs)

In [ ]:
import zarr
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

rng = np.random.default_rng(42)

# 先各抽一部分 dataset，避免第一次实验就扫完整 82 GB
sampled_paths = []

for prefix in ["44b6", "6bba"]:
    candidates = [
        p for p in train_zarrs
        if p.stem.startswith(prefix)
    ]

    n = min(20, len(candidates))

    sampled_paths.extend(
        rng.choice(
            candidates,
            size=n,
            replace=False,
        ).tolist()
    )

len(sampled_paths)

In [ ]:
gt_peak_records = []

# local 3D patch radius
rz = 2
ry = 6
rx = 6

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    # 只读 GT，不加载整个 4D image
    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    # 每个 dataset 最多抽 50 个 GT nodes
    n_sample = min(50, nodes.height)

    sample_idx = rng.choice(
        nodes.height,
        size=n_sample,
        replace=False,
    )

    sampled_nodes = nodes[
        sample_idx.tolist()
    ]

    # 直接访问 zarr，避免把整个视频读入内存
    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    T, Z, Y, X = image.shape

    for row in sampled_nodes.iter_rows(named=True):

        t = int(row["t"])
        z = int(row["z"])
        y = int(row["y"])
        x = int(row["x"])

        z0 = max(0, z - rz)
        z1 = min(Z, z + rz + 1)

        y0 = max(0, y - ry)
        y1 = min(Y, y + ry + 1)

        x0 = max(0, x - rx)
        x1 = min(X, x + rx + 1)

        patch = np.asarray(
            image[
                t,
                z0:z1,
                y0:y1,
                x0:x1,
            ],
            dtype=np.float32,
        )

        center_intensity = float(
            image[t, z, y, x]
        )

        flat = patch.ravel()

        # GT centroid intensity 在局部 patch 中的 percentile
        center_percentile = (
            np.mean(
                flat <= center_intensity
            )
        )

        local_max = float(
            patch.max()
        )

        gt_peak_records.append({
            "dataset": name,
            "prefix": prefix,
            "t": t,
            "z": z,
            "y": y,
            "x": x,
            "center_intensity": center_intensity,
            "local_max_intensity": local_max,
            "center_percentile": center_percentile,
            "center_is_local_max":
                center_intensity == local_max,
        })

gt_peak_df = pd.DataFrame(
    gt_peak_records
)

print("Sampled GT nodes:", len(gt_peak_df))

gt_peak_df.head()

In [ ]:
gt_peak_df.groupby("prefix")[
    "center_percentile"
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
    ]
)

GT centroid 通常位于局部高亮区域，但几乎从不恰好等于最亮 voxel
	​


In [ ]:
gt_peak_df.groupby("prefix")[
    "center_is_local_max"
].mean()

98%–99.5% 的 GT centroid 都不是 patch 中最亮的那个 voxel

In [ ]:
peak_distance_records = []

rz = 2
ry = 6
rx = 6

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    # 使用和前面相同的随机逻辑重新抽 50 个；
    # 这里主要看总体分布
    n_sample = min(50, nodes.height)

    sample_idx = rng.choice(
        nodes.height,
        size=n_sample,
        replace=False,
    )

    sampled_nodes = nodes[
        sample_idx.tolist()
    ]

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    T, Z, Y, X = image.shape

    for row in sampled_nodes.iter_rows(named=True):

        t = int(row["t"])
        z = int(row["z"])
        y = int(row["y"])
        x = int(row["x"])

        z0 = max(0, z - rz)
        z1 = min(Z, z + rz + 1)

        y0 = max(0, y - ry)
        y1 = min(Y, y + ry + 1)

        x0 = max(0, x - rx)
        x1 = min(X, x + rx + 1)

        patch = np.asarray(
            image[
                t,
                z0:z1,
                y0:y1,
                x0:x1,
            ],
            dtype=np.float32,
        )

        # location of brightest voxel inside patch
        local_index = np.unravel_index(
            np.argmax(patch),
            patch.shape,
        )

        peak_z = z0 + local_index[0]
        peak_y = y0 + local_index[1]
        peak_x = x0 + local_index[2]

        delta_voxel = np.array(
            [
                peak_z - z,
                peak_y - y,
                peak_x - x,
            ],
            dtype=float,
        )

        delta_um = delta_voxel * scale

        peak_distance_um = np.linalg.norm(
            delta_um
        )

        peak_distance_records.append({
            "dataset": name,
            "prefix": prefix,
            "t": t,
            "gt_z": z,
            "gt_y": y,
            "gt_x": x,
            "peak_z": peak_z,
            "peak_y": peak_y,
            "peak_x": peak_x,
            "dz_voxel": delta_voxel[0],
            "dy_voxel": delta_voxel[1],
            "dx_voxel": delta_voxel[2],
            "peak_distance_um":
                peak_distance_um,
        })

peak_distance_df = pd.DataFrame(
    peak_distance_records
)

In [ ]:
peak_distance_df.groupby("prefix")[
    "peak_distance_um"
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

In [ ]:
for prefix in ["44b6", "6bba"]:

    df = peak_distance_df[
        peak_distance_df["prefix"] == prefix
    ]

    print(f"\n{prefix}")

    for radius in [0.5, 1, 2, 3, 4]:
        fraction = (
            df["peak_distance_um"] <= radius
        ).mean()

        print(
            f"Local max within {radius} µm: "
            f"{fraction:.3f}"
        )

44b6 的亮峰通常离 GT centroid 更近​

而 6bba 的 centroid 与 intensity peak 的对应关系更弱

In [ ]:
from skimage.feature import peak_local_max

nearest_peak_records = []

# 用稍微大一点的窗口，避免之前 4.74 µm 的硬截断
rz = 4
ry = 10
rx = 10

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    n_sample = min(50, nodes.height)

    sample_idx = rng.choice(
        nodes.height,
        size=n_sample,
        replace=False,
    )

    sampled_nodes = nodes[
        sample_idx.tolist()
    ]

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    T, Z, Y, X = image.shape

    for row in sampled_nodes.iter_rows(named=True):

        t = int(row["t"])
        z = int(row["z"])
        y = int(row["y"])
        x = int(row["x"])

        z0 = max(0, z - rz)
        z1 = min(Z, z + rz + 1)

        y0 = max(0, y - ry)
        y1 = min(Y, y + ry + 1)

        x0 = max(0, x - rx)
        x1 = min(X, x + rx + 1)

        patch = np.asarray(
            image[
                t,
                z0:z1,
                y0:y1,
                x0:x1,
            ],
            dtype=np.float32,
        )

        # Local maxima inside the patch
        peaks = peak_local_max(
            patch,
            min_distance=1,
            exclude_border=False,
        )

        if len(peaks) == 0:
            continue

        # Convert local patch coordinates back to global voxel coordinates
        peak_global = peaks + np.array(
            [z0, y0, x0]
        )

        gt_coord = np.array(
            [z, y, x]
        )

        delta_voxel = (
            peak_global - gt_coord
        )

        delta_um = (
            delta_voxel * scale
        )

        distances_um = np.linalg.norm(
            delta_um,
            axis=1,
        )

        nearest_distance = float(
            distances_um.min()
        )

        nearest_peak_records.append({
            "dataset": name,
            "prefix": prefix,
            "t": t,
            "nearest_peak_distance_um":
                nearest_distance,
            "n_local_peaks":
                len(peaks),
        })

nearest_peak_df = pd.DataFrame(
    nearest_peak_records
)

In [ ]:
nearest_peak_df.groupby("prefix")[
    "nearest_peak_distance_um"
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

In [ ]:
for prefix in ["44b6", "6bba"]:

    df = nearest_peak_df[
        nearest_peak_df["prefix"] == prefix
    ]

    print(f"\n{prefix}")

    for radius in [0.5, 1, 2, 3, 4]:

        fraction = (
            df["nearest_peak_distance_um"]
            <= radius
        ).mean()

        print(
            f"Nearest local peak within {radius} µm: "
            f"{fraction:.3f}"
        )

单纯的 3D local-peak detector，在“GT 附近有没有 candidate”这个 recall 问题上，很有潜力

In [ ]:
peak_count_records = []

timepoints = [0, 24, 49, 74, 99]

# 用局部 frame intensity percentile 做 threshold
percentile_thresholds = [
    90,
    95,
    97,
    99,
    99.5,
]

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_nodes_t = (
            nodes
            .filter(pl.col("t") == t)
            .height
        )

        for percentile in percentile_thresholds:

            threshold = np.percentile(
                frame,
                percentile,
            )

            peaks = peak_local_max(
                frame,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            peak_count_records.append({
                "dataset": name,
                "prefix": prefix,
                "t": t,
                "percentile": percentile,
                "n_peaks": len(peaks),
                "n_gt_nodes": gt_nodes_t,
            })

peak_count_df = pd.DataFrame(
    peak_count_records
)

peak_count_df.head()

In [ ]:
peak_count_summary = (
    peak_count_df
    .groupby(
        ["prefix", "percentile"]
    )
    .agg(
        median_peaks=("n_peaks", "median"),
        mean_peaks=("n_peaks", "mean"),
        p90_peaks=(
            "n_peaks",
            lambda x: x.quantile(0.90),
        ),
        median_gt_nodes=("n_gt_nodes", "median"),
    )
)

peak_count_summary

In [ ]:
from scipy.spatial import cKDTree

detection_tradeoff_records = []

timepoints = [0, 24, 49, 74, 99]

percentile_thresholds = [
    90,
    95,
    97,
    99,
    99.5,
]

match_radii_um = [
    1,
    2,
    3,
    4,
]

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        # Convert GT coordinates to physical microns
        if len(gt_coords) > 0:
            gt_coords_um = (
                gt_coords * scale
            )
        else:
            gt_coords_um = np.empty(
                (0, 3),
                dtype=float,
            )

        for percentile in percentile_thresholds:

            threshold = np.percentile(
                frame,
                percentile,
            )

            peaks = peak_local_max(
                frame,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            if len(peaks) > 0:
                peak_coords_um = (
                    peaks.astype(float)
                    * scale
                )

                tree = cKDTree(
                    peak_coords_um
                )

                if len(gt_coords_um) > 0:
                    nearest_distances, _ = (
                        tree.query(
                            gt_coords_um,
                            k=1,
                        )
                    )
                else:
                    nearest_distances = (
                        np.array([])
                    )

            else:
                nearest_distances = np.full(
                    len(gt_coords_um),
                    np.inf,
                )

            record = {
                "dataset": name,
                "prefix": prefix,
                "t": t,
                "percentile": percentile,
                "n_peaks": len(peaks),
                "n_gt": len(gt_coords_um),
            }

            for radius in match_radii_um:

                if len(gt_coords_um) == 0:
                    coverage = np.nan
                else:
                    coverage = (
                        nearest_distances
                        <= radius
                    ).mean()

                record[
                    f"coverage_{radius}um"
                ] = coverage

            detection_tradeoff_records.append(
                record
            )

detection_tradeoff_df = pd.DataFrame(
    detection_tradeoff_records
)

In [ ]:
tradeoff_summary = (
    detection_tradeoff_df
    .groupby(
        ["prefix", "percentile"]
    )
    .agg(
        median_candidates=(
            "n_peaks",
            "median",
        ),

        coverage_1um=(
            "coverage_1um",
            "mean",
        ),

        coverage_2um=(
            "coverage_2um",
            "mean",
        ),

        coverage_3um=(
            "coverage_3um",
            "mean",
        ),

        coverage_4um=(
            "coverage_4um",
            "mean",
        ),
    )
)

tradeoff_summary

它更适合作为 high-recall proposal generator，而不是最终 detector

只按绝对亮度 percentile 筛 peak，反而很容易把 44b6 的 GT peak 一起删掉

local contrast not global intensity rank​

In [ ]:
peak_feature_records = []

timepoints = [0, 24, 49, 74, 99]

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:
            gt_coords_um = (
                gt_coords * scale
            )
        else:
            gt_coords_um = np.empty(
                (0, 3)
            )

        threshold = np.percentile(
            frame,
            95,
        )

        peaks = peak_local_max(
            frame,
            min_distance=1,
            threshold_abs=threshold,
            exclude_border=False,
        )

        if len(peaks) == 0:
            continue

        # GT matching tree
        if len(gt_coords_um) > 0:
            gt_tree = cKDTree(
                gt_coords_um
            )

        for z, y, x in peaks:

            peak_pos_um = (
                np.array(
                    [z, y, x],
                    dtype=float,
                )
                * scale
            )

            if len(gt_coords_um) > 0:
                gt_distance, _ = (
                    gt_tree.query(
                        peak_pos_um,
                        k=1,
                    )
                )
            else:
                gt_distance = np.inf

            # small local patch around peak
            z0 = max(0, z - 2)
            z1 = min(frame.shape[0], z + 3)

            y0 = max(0, y - 4)
            y1 = min(frame.shape[1], y + 5)

            x0 = max(0, x - 4)
            x1 = min(frame.shape[2], x + 5)

            patch = frame[
                z0:z1,
                y0:y1,
                x0:x1,
            ]

            center_intensity = float(
                frame[z, y, x]
            )

            local_median = float(
                np.median(patch)
            )

            local_mean = float(
                np.mean(patch)
            )

            local_std = float(
                np.std(patch)
            )

            contrast = (
                center_intensity
                - local_median
            )

            zscore = (
                center_intensity
                - local_mean
            ) / (
                local_std + 1e-8
            )

            peak_feature_records.append({
                "dataset": name,
                "prefix": prefix,
                "t": t,

                "gt_distance_um":
                    gt_distance,

                # 3 µm 内暂时视为 GT-associated proposal
                "near_gt":
                    gt_distance <= 3.0,

                "intensity":
                    center_intensity,

                "local_contrast":
                    contrast,

                "local_zscore":
                    zscore,
            })

peak_feature_df = pd.DataFrame(
    peak_feature_records
)

print(
    peak_feature_df.groupby(
        ["prefix", "near_gt"]
    ).size()
)

In [ ]:
peak_feature_df.groupby(
    ["prefix", "near_gt"]
)[
    [
        "local_contrast",
        "local_zscore",
    ]
].median()

局部对比度确实有信号，44b6 的 intensity/contrast signal 比 6bba 更直接​

但 near_gt=False 绝对不能叫 negative/background

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

for prefix in ["44b6", "6bba"]:

    df = peak_feature_df[
        peak_feature_df["prefix"] == prefix
    ]

    y = df["near_gt"].astype(int)

    print(f"\n{prefix}")
    print(
        f"GT-associated proposal rate: "
        f"{y.mean():.4f}"
    )

    for feature in [
        "intensity",
        "local_contrast",
        "local_zscore",
    ]:

        scores = df[feature]

        auc = roc_auc_score(
            y,
            scores,
        )

        ap = average_precision_score(
            y,
            scores,
        )

        print(
            f"{feature:16s} "
            f"ROC-AUC={auc:.4f} "
            f"AP={ap:.4f}"
        )

单纯亮度和局部对比度不足以筛选 cell proposals​

In [ ]:
from scipy.ndimage import gaussian_laplace

blob_feature_records = []

timepoints = [0, 24, 49, 74, 99]

# Physical blob scales in microns
blob_scales_um = [
    1.0,
    1.5,
    2.0,
]

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:
            gt_coords_um = (
                gt_coords * scale
            )
            gt_tree = cKDTree(
                gt_coords_um
            )
        else:
            gt_coords_um = np.empty(
                (0, 3),
                dtype=float,
            )
            gt_tree = None

        # Same proposal rule as previous experiment
        threshold = np.percentile(
            frame,
            95,
        )

        peaks = peak_local_max(
            frame,
            min_distance=1,
            threshold_abs=threshold,
            exclude_border=False,
        )

        if len(peaks) == 0:
            continue

        # Compute LoG response maps ONCE per frame
        log_maps = {}

        for sigma_um in blob_scales_um:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            # Scale-normalized approximate LoG response
            response *= sigma_um ** 2

            log_maps[sigma_um] = response

        for z, y, x in peaks:

            peak_pos_um = (
                np.array(
                    [z, y, x],
                    dtype=float,
                )
                * scale
            )

            if gt_tree is not None:
                gt_distance, _ = (
                    gt_tree.query(
                        peak_pos_um,
                        k=1,
                    )
                )
            else:
                gt_distance = np.inf

            responses = {
                sigma_um:
                    float(
                        log_maps[sigma_um][z, y, x]
                    )
                for sigma_um in blob_scales_um
            }

            blob_feature_records.append({
                "dataset": name,
                "prefix": prefix,
                "t": t,

                "near_gt":
                    gt_distance <= 3.0,

                "log_1um":
                    responses[1.0],

                "log_1_5um":
                    responses[1.5],

                "log_2um":
                    responses[2.0],

                "max_log":
                    max(responses.values()),
            })

blob_feature_df = pd.DataFrame(
    blob_feature_records
)

blob_feature_df.head()

In [ ]:
for prefix in ["44b6", "6bba"]:

    df = blob_feature_df[
        blob_feature_df["prefix"] == prefix
    ]

    y = df["near_gt"].astype(int)

    print(f"\n{prefix}")

    for feature in [
        "log_1um",
        "log_1_5um",
        "log_2um",
        "max_log",
    ]:

        auc = roc_auc_score(
            y,
            df[feature],
        )

        ap = average_precision_score(
            y,
            df[feature],
        )

        print(
            f"{feature:10s} "
            f"ROC-AUC={auc:.4f} "
            f"AP={ap:.4f}"
        )

cell-like 3D shape 比单纯“亮不亮”更有判别力

GT-associated structures 本身似乎具有一个偏大的典型 spatial scale

In [ ]:
log_scale_records = []

log_scales_um = [
    0.75,
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
    3.5,
    4.0,
]

timepoints = [0, 24, 49, 74, 99]

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:

            gt_coords_um = (
                gt_coords * scale
            )

            gt_tree = cKDTree(
                gt_coords_um
            )

        else:

            gt_tree = None

        threshold = np.percentile(
            frame,
            95,
        )

        peaks = peak_local_max(
            frame,
            min_distance=1,
            threshold_abs=threshold,
            exclude_border=False,
        )

        if len(peaks) == 0:
            continue

        peak_coords_um = (
            peaks.astype(float)
            * scale
        )

        if gt_tree is not None:

            gt_distances, _ = (
                gt_tree.query(
                    peak_coords_um,
                    k=1,
                )
            )

        else:

            gt_distances = np.full(
                len(peaks),
                np.inf,
            )

        near_gt = (
            gt_distances <= 3.0
        )

        # Evaluate each LoG scale
        for sigma_um in log_scales_um:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            response *= sigma_um ** 2

            peak_response = response[
                peaks[:, 0],
                peaks[:, 1],
                peaks[:, 2],
            ]

            for label, score in zip(
                near_gt,
                peak_response,
            ):

                log_scale_records.append({
                    "dataset": name,
                    "prefix": prefix,
                    "t": t,
                    "sigma_um": sigma_um,
                    "near_gt": bool(label),
                    "log_response": float(score),
                })

log_scale_df = pd.DataFrame(
    log_scale_records
)

In [ ]:
log_scale_summary = []

for (prefix, sigma_um), df in (
    log_scale_df
    .groupby(
        ["prefix", "sigma_um"]
    )
):

    y = df["near_gt"].astype(int)

    auc = roc_auc_score(
        y,
        df["log_response"],
    )

    ap = average_precision_score(
        y,
        df["log_response"],
    )

    log_scale_summary.append({
        "prefix": prefix,
        "sigma_um": sigma_um,
        "roc_auc": auc,
        "ap": ap,
    })

log_scale_summary = pd.DataFrame(
    log_scale_summary
)

log_scale_summary

44b6 已经出现清楚的峰值：

σ≈2.0–2.5 μm
	​


In [ ]:
extended_scales_um = [
    4.5,
    5.0,
    6.0,
    7.0,
    8.0,
]

extended_scores = {
    sigma: {
        "labels": [],
        "scores": [],
    }
    for sigma in extended_scales_um
}

sixbba_paths = [
    p for p in sampled_paths
    if p.stem.startswith("6bba")
]

timepoints = [0, 24, 49, 74, 99]

for zarr_path in tqdm(sixbba_paths):

    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:

            gt_coords_um = (
                gt_coords * scale
            )

            gt_tree = cKDTree(
                gt_coords_um
            )

        else:
            gt_tree = None

        threshold = np.percentile(
            frame,
            95,
        )

        peaks = peak_local_max(
            frame,
            min_distance=1,
            threshold_abs=threshold,
            exclude_border=False,
        )

        if len(peaks) == 0:
            continue

        peak_coords_um = (
            peaks.astype(float)
            * scale
        )

        if gt_tree is not None:

            gt_distances, _ = (
                gt_tree.query(
                    peak_coords_um,
                    k=1,
                )
            )

            near_gt = (
                gt_distances <= 3.0
            )

        else:

            near_gt = np.zeros(
                len(peaks),
                dtype=bool,
            )

        for sigma_um in extended_scales_um:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            # Positive scaling does not affect within-scale AUC/AP ranking
            response *= sigma_um ** 2

            peak_response = response[
                peaks[:, 0],
                peaks[:, 1],
                peaks[:, 2],
            ]

            extended_scores[
                sigma_um
            ]["labels"].append(
                near_gt
            )

            extended_scores[
                sigma_um
            ]["scores"].append(
                peak_response
            )

In [ ]:
extended_summary = []

for sigma_um in extended_scales_um:

    y = np.concatenate(
        extended_scores[
            sigma_um
        ]["labels"]
    ).astype(int)

    scores = np.concatenate(
        extended_scores[
            sigma_um
        ]["scores"]
    )

    extended_summary.append({
        "prefix": "6bba",
        "sigma_um": sigma_um,
        "roc_auc": roc_auc_score(
            y,
            scores,
        ),
        "ap": average_precision_score(
            y,
            scores,
        ),
    })

extended_summary = pd.DataFrame(
    extended_summary
)

extended_summary

44b6:σ≈2.0 μm
	​

6bba:σ≈4.0–4.5 μm
	

In [ ]:
log_candidate_records = []

timepoints = [0, 24, 49, 74, 99]

percentile_thresholds = [
    90,
    95,
    97,
    99,
    99.5,
]

match_radii_um = [
    1,
    2,
    3,
    4,
]

best_sigma = {
    "44b6": 2.0,
    "6bba": 4.5,
}

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    sigma_um = best_sigma[prefix]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    sigma_voxel = (
        sigma_um / scale
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:

            gt_coords_um = (
                gt_coords * scale
            )

        else:

            gt_coords_um = np.empty(
                (0, 3),
                dtype=float,
            )

        # Domain-specific 3D LoG response
        log_response = -gaussian_laplace(
            frame,
            sigma=tuple(sigma_voxel),
        )

        log_response *= sigma_um ** 2

        for percentile in percentile_thresholds:

            threshold = np.percentile(
                log_response,
                percentile,
            )

            peaks = peak_local_max(
                log_response,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            if len(peaks) > 0:

                peak_coords_um = (
                    peaks.astype(float)
                    * scale
                )

                tree = cKDTree(
                    peak_coords_um
                )

                if len(gt_coords_um) > 0:

                    nearest_distances, _ = (
                        tree.query(
                            gt_coords_um,
                            k=1,
                        )
                    )

                else:

                    nearest_distances = np.array([])

            else:

                nearest_distances = np.full(
                    len(gt_coords_um),
                    np.inf,
                )

            record = {
                "dataset": name,
                "prefix": prefix,
                "t": t,
                "sigma_um": sigma_um,
                "percentile": percentile,
                "n_candidates": len(peaks),
                "n_gt": len(gt_coords_um),
            }

            for radius in match_radii_um:

                if len(gt_coords_um) == 0:

                    coverage = np.nan

                else:

                    coverage = (
                        nearest_distances
                        <= radius
                    ).mean()

                record[
                    f"coverage_{radius}um"
                ] = coverage

            log_candidate_records.append(
                record
            )

log_candidate_df = pd.DataFrame(
    log_candidate_records
)

In [ ]:
log_candidate_summary = (
    log_candidate_df
    .groupby(
        ["prefix", "percentile"]
    )
    .agg(
        median_candidates=(
            "n_candidates",
            "median",
        ),

        coverage_1um=(
            "coverage_1um",
            "mean",
        ),

        coverage_2um=(
            "coverage_2um",
            "mean",
        ),

        coverage_3um=(
            "coverage_3um",
            "mean",
        ),

        coverage_4um=(
            "coverage_4um",
            "mean",
        ),
    )
)

log_candidate_summary

LoG blob proposals ≫ raw-intensity peak proposals​

In [ ]:
log_low_threshold_records = []

timepoints = [0, 24, 49, 74, 99]

percentile_thresholds = [
    50,
    60,
    70,
    80,
    85,
    90,
    95,
]

match_radii_um = [
    1,
    2,
    3,
    4,
]

best_sigma = {
    "44b6": 2.0,
    "6bba": 4.5,
}

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    sigma_um = best_sigma[prefix]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    sigma_voxel = (
        sigma_um / scale
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:
            gt_coords_um = (
                gt_coords * scale
            )
        else:
            gt_coords_um = np.empty(
                (0, 3),
                dtype=float,
            )

        log_response = -gaussian_laplace(
            frame,
            sigma=tuple(sigma_voxel),
        )

        log_response *= sigma_um ** 2

        for percentile in percentile_thresholds:

            threshold = np.percentile(
                log_response,
                percentile,
            )

            peaks = peak_local_max(
                log_response,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            if len(peaks) > 0:

                peak_coords_um = (
                    peaks.astype(float)
                    * scale
                )

                tree = cKDTree(
                    peak_coords_um
                )

                if len(gt_coords_um) > 0:

                    nearest_distances, _ = (
                        tree.query(
                            gt_coords_um,
                            k=1,
                        )
                    )

                else:
                    nearest_distances = np.array([])

            else:

                nearest_distances = np.full(
                    len(gt_coords_um),
                    np.inf,
                )

            record = {
                "dataset": name,
                "prefix": prefix,
                "t": t,
                "percentile": percentile,
                "n_candidates": len(peaks),
                "n_gt": len(gt_coords_um),
            }

            for radius in match_radii_um:

                if len(gt_coords_um) == 0:
                    coverage = np.nan
                else:
                    coverage = (
                        nearest_distances
                        <= radius
                    ).mean()

                record[
                    f"coverage_{radius}um"
                ] = coverage

            log_low_threshold_records.append(
                record
            )

log_low_threshold_df = pd.DataFrame(
    log_low_threshold_records
)

In [ ]:
log_low_threshold_summary = (
    log_low_threshold_df
    .groupby(
        ["prefix", "percentile"]
    )
    .agg(
        median_candidates=(
            "n_candidates",
            "median",
        ),
        mean_candidates=(
            "n_candidates",
            "mean",
        ),
        coverage_1um=(
            "coverage_1um",
            "mean",
        ),
        coverage_2um=(
            "coverage_2um",
            "mean",
        ),
        coverage_3um=(
            "coverage_3um",
            "mean",
        ),
        coverage_4um=(
            "coverage_4um",
            "mean",
        ),
    )
)

log_low_threshold_summary

In [ ]:
proposal_scales_um = [
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    4.5,
]

timepoints = [0, 24, 49, 74, 99]

proposal_scale_records = []

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) > 0:
            gt_coords_um = (
                gt_coords * scale
            )
        else:
            gt_coords_um = np.empty(
                (0, 3),
                dtype=float,
            )

        for sigma_um in proposal_scales_um:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            response *= sigma_um ** 2

            # very permissive threshold
            threshold = np.percentile(
                response,
                50,
            )

            peaks = peak_local_max(
                response,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            if len(peaks) > 0:

                peak_coords_um = (
                    peaks.astype(float)
                    * scale
                )

                tree = cKDTree(
                    peak_coords_um
                )

                if len(gt_coords_um) > 0:

                    nearest_distances, _ = (
                        tree.query(
                            gt_coords_um,
                            k=1,
                        )
                    )

                else:

                    nearest_distances = np.array([])

            else:

                nearest_distances = np.full(
                    len(gt_coords_um),
                    np.inf,
                )

            proposal_scale_records.append({
                "dataset": name,
                "prefix": prefix,
                "t": t,
                "sigma_um": sigma_um,

                "n_candidates":
                    len(peaks),

                "coverage_1um":
                    np.mean(
                        nearest_distances <= 1
                    )
                    if len(nearest_distances)
                    else np.nan,

                "coverage_2um":
                    np.mean(
                        nearest_distances <= 2
                    )
                    if len(nearest_distances)
                    else np.nan,

                "coverage_3um":
                    np.mean(
                        nearest_distances <= 3
                    )
                    if len(nearest_distances)
                    else np.nan,

                "coverage_4um":
                    np.mean(
                        nearest_distances <= 4
                    )
                    if len(nearest_distances)
                    else np.nan,
            })

proposal_scale_df = pd.DataFrame(
    proposal_scale_records
)

In [ ]:
proposal_scale_summary = (
    proposal_scale_df
    .groupby(
        ["prefix", "sigma_um"]
    )
    .agg(
        median_candidates=(
            "n_candidates",
            "median",
        ),

        mean_candidates=(
            "n_candidates",
            "mean",
        ),

        coverage_1um=(
            "coverage_1um",
            "mean",
        ),

        coverage_2um=(
            "coverage_2um",
            "mean",
        ),

        coverage_3um=(
            "coverage_3um",
            "mean",
        ),

        coverage_4um=(
            "coverage_4um",
            "mean",
        ),
    )
)

proposal_scale_summary

In [ ]:
rng_control = np.random.default_rng(12345)

random_control_records = []

random_repeats = 3

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    _, Z, Y, X = image.shape

    for t in [0, 24, 49, 74, 99]:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) == 0:
            continue

        gt_coords_um = (
            gt_coords * scale
        )

        for sigma_um in proposal_scales_um:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            response *= sigma_um ** 2

            threshold = np.percentile(
                response,
                50,
            )

            peaks = peak_local_max(
                response,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            n_candidates = len(peaks)

            if n_candidates == 0:
                continue

            # -------------------------
            # Actual LoG coverage
            # -------------------------

            peak_coords_um = (
                peaks.astype(float)
                * scale
            )

            peak_tree = cKDTree(
                peak_coords_um
            )

            log_distances, _ = (
                peak_tree.query(
                    gt_coords_um,
                    k=1,
                )
            )

            actual_coverage = {
                radius:
                    np.mean(
                        log_distances <= radius
                    )
                for radius in [1, 2, 3, 4]
            }

            # -------------------------
            # Random matched controls
            # -------------------------

            for repeat in range(
                random_repeats
            ):

                random_coords = np.column_stack([
                    rng_control.integers(
                        0, Z,
                        size=n_candidates,
                    ),
                    rng_control.integers(
                        0, Y,
                        size=n_candidates,
                    ),
                    rng_control.integers(
                        0, X,
                        size=n_candidates,
                    ),
                ]).astype(float)

                random_coords_um = (
                    random_coords * scale
                )

                random_tree = cKDTree(
                    random_coords_um
                )

                random_distances, _ = (
                    random_tree.query(
                        gt_coords_um,
                        k=1,
                    )
                )

                record = {
                    "dataset": name,
                    "prefix": prefix,
                    "t": t,
                    "sigma_um": sigma_um,
                    "n_candidates":
                        n_candidates,
                    "repeat":
                        repeat,
                }

                for radius in [
                    1, 2, 3, 4
                ]:

                    random_cov = np.mean(
                        random_distances
                        <= radius
                    )

                    record[
                        f"log_coverage_{radius}um"
                    ] = actual_coverage[
                        radius
                    ]

                    record[
                        f"random_coverage_{radius}um"
                    ] = random_cov

                    record[
                        f"gain_{radius}um"
                    ] = (
                        actual_coverage[
                            radius
                        ]
                        - random_cov
                    )

                random_control_records.append(
                    record
                )

random_control_df = pd.DataFrame(
    random_control_records
)

In [ ]:
random_control_summary = (
    random_control_df
    .groupby(
        ["prefix", "sigma_um"]
    )
    .agg(
        median_candidates=(
            "n_candidates",
            "median",
        ),

        log_coverage_2um=(
            "log_coverage_2um",
            "mean",
        ),

        random_coverage_2um=(
            "random_coverage_2um",
            "mean",
        ),

        gain_2um=(
            "gain_2um",
            "mean",
        ),

        log_coverage_3um=(
            "log_coverage_3um",
            "mean",
        ),

        random_coverage_3um=(
            "random_coverage_3um",
            "mean",
        ),

        gain_3um=(
            "gain_3um",
            "mean",
        ),

        log_coverage_4um=(
            "log_coverage_4um",
            "mean",
        ),

        random_coverage_4um=(
            "random_coverage_4um",
            "mean",
        ),

        gain_4um=(
            "gain_4um",
            "mean",
        ),
    )
)

random_control_summary

44b6:
proposal ≈ 2–2.5 µm
scoring  ≈ 2–2.5 µm

6bba:
proposal ≈ 3–3.5 µm
context/scoring ≈ 4–4.5 µm

In [ ]:
fixed_budget_records = []

timepoints = [0, 24, 49, 74, 99]

proposal_scales_um = [
    1.5,
    2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    4.5,
]

candidate_budgets = [
    50,
    100,
    200,
    500,
    1000,
]

for zarr_path in tqdm(sampled_paths):

    name = zarr_path.stem
    prefix = name.split("_")[0]

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) == 0:
            continue

        gt_coords_um = (
            gt_coords * scale
        )

        for sigma_um in proposal_scales_um:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            response *= sigma_um ** 2

            # Generate a permissive set of local maxima
            threshold = np.percentile(
                response,
                50,
            )

            peaks = peak_local_max(
                response,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            if len(peaks) == 0:
                continue

            # Rank local maxima by LoG response
            peak_scores = response[
                peaks[:, 0],
                peaks[:, 1],
                peaks[:, 2],
            ]

            order = np.argsort(
                peak_scores
            )[::-1]

            peaks = peaks[order]

            for budget in candidate_budgets:

                k = min(
                    budget,
                    len(peaks),
                )

                selected = peaks[:k]

                selected_um = (
                    selected.astype(float)
                    * scale
                )

                tree = cKDTree(
                    selected_um
                )

                distances, _ = tree.query(
                    gt_coords_um,
                    k=1,
                )

                fixed_budget_records.append({
                    "dataset": name,
                    "prefix": prefix,
                    "t": t,
                    "sigma_um": sigma_um,
                    "budget": budget,
                    "actual_candidates": k,

                    "coverage_2um":
                        np.mean(
                            distances <= 2
                        ),

                    "coverage_3um":
                        np.mean(
                            distances <= 3
                        ),

                    "coverage_4um":
                        np.mean(
                            distances <= 4
                        ),
                })

fixed_budget_df = pd.DataFrame(
    fixed_budget_records
)

In [ ]:
fixed_budget_summary = (
    fixed_budget_df
    .groupby(
        ["prefix", "sigma_um", "budget"]
    )
    .agg(
        mean_actual_candidates=(
            "actual_candidates",
            "mean",
        ),

        coverage_2um=(
            "coverage_2um",
            "mean",
        ),

        coverage_3um=(
            "coverage_3um",
            "mean",
        ),

        coverage_4um=(
            "coverage_4um",
            "mean",
        ),
    )
    .reset_index()
)

fixed_budget_summary

In [ ]:
for prefix in ["44b6", "6bba"]:

    print(f"\n===== {prefix} =====")

    display(
        fixed_budget_summary[
            fixed_budget_summary[
                "prefix"
            ] == prefix
        ].pivot(
            index="sigma_um",
            columns="budget",
            values="coverage_3um",
        )
    )

1. 44b6：σ≈2.5 µm 是最稳定的工作点
2. 6bba budget 越小，最佳 scale 越大​
3. 单尺度 LoG 不足以作为最终 proposal generator​

multi-scale proposals
+
learned 3D detector / score map
+
possibly centroid refinement

In [ ]:
multiscale_pairs = [
    (1.5, 3.0),
    (1.5, 3.5),
    (2.0, 3.5),
    (2.5, 4.0),
    (3.0, 4.5),
]

timepoints = [0, 24, 49, 74, 99]

multiscale_records = []

sixbba_paths = [
    p for p in sampled_paths
    if p.stem.startswith("6bba")
]

for zarr_path in tqdm(sixbba_paths):

    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    scale = np.array(
        ds_meta.scale,
        dtype=float,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    group = zarr.open_group(
        zarr_path,
        mode="r",
    )

    image = group["0"]

    for t in timepoints:

        frame = np.asarray(
            image[t],
            dtype=np.float32,
        )

        gt_t = nodes.filter(
            pl.col("t") == t
        )

        gt_coords = np.array(
            gt_t.select(
                ["z", "y", "x"]
            ).rows(),
            dtype=float,
        )

        if len(gt_coords) == 0:
            continue

        gt_coords_um = gt_coords * scale

        # Cache top proposals for every scale
        scale_peaks = {}

        needed_scales = sorted(
            set(
                s
                for pair in multiscale_pairs
                for s in pair
            )
        )

        for sigma_um in needed_scales:

            sigma_voxel = (
                sigma_um / scale
            )

            response = -gaussian_laplace(
                frame,
                sigma=tuple(sigma_voxel),
            )

            response *= sigma_um ** 2

            threshold = np.percentile(
                response,
                50,
            )

            peaks = peak_local_max(
                response,
                min_distance=1,
                threshold_abs=threshold,
                exclude_border=False,
            )

            scores = response[
                peaks[:, 0],
                peaks[:, 1],
                peaks[:, 2],
            ]

            order = np.argsort(
                scores
            )[::-1]

            scale_peaks[sigma_um] = (
                peaks[order]
            )

        for sigma_a, sigma_b in multiscale_pairs:

            # 100 from each scale = max 200 raw candidates
            a = scale_peaks[
                sigma_a
            ][:100]

            b = scale_peaks[
                sigma_b
            ][:100]

            combined = np.vstack(
                [a, b]
            )

            # Remove exact duplicate voxel coordinates
            combined = np.unique(
                combined,
                axis=0,
            )

            combined_um = (
                combined.astype(float)
                * scale
            )

            tree = cKDTree(
                combined_um
            )

            distances, _ = tree.query(
                gt_coords_um,
                k=1,
            )

            multiscale_records.append({
                "dataset": name,
                "t": t,
                "sigma_a": sigma_a,
                "sigma_b": sigma_b,
                "n_candidates":
                    len(combined),

                "coverage_2um":
                    np.mean(
                        distances <= 2
                    ),

                "coverage_3um":
                    np.mean(
                        distances <= 3
                    ),

                "coverage_4um":
                    np.mean(
                        distances <= 4
                    ),
            })

multiscale_df = pd.DataFrame(
    multiscale_records
)

In [ ]:
multiscale_summary = (
    multiscale_df
    .groupby(
        ["sigma_a", "sigma_b"]
    )
    .agg(
        mean_candidates=(
            "n_candidates",
            "mean",
        ),
        coverage_2um=(
            "coverage_2um",
            "mean",
        ),
        coverage_3um=(
            "coverage_3um",
            "mean",
        ),
        coverage_4um=(
            "coverage_4um",
            "mean",
        ),
    )
    .reset_index()
)

multiscale_summary


6bba 的 multi-scale LoG proposals 确实存在明显互补性
	​


1. GT centroid 通常位于局部亮区域
2. centroid 几乎从不精确等于 brightest voxel
3. GT 周围通常存在 local peak
4. raw intensity peaks candidate burden 极高
5. global intensity threshold 会严重伤害 GT coverage
6. local contrast 单独判别力弱
7. 3D LoG morphology 明显优于 raw intensity
8. 44b6 / 6bba 最优 spatial scales 不同
9. candidate-density random control 证明：
   极小尺度的高 coverage 很大部分是密度假象
10. 合理尺度的 LoG 确实具有真实 localization signal
11. 单尺度 LoG recall 仍不足
12. 6bba multi-scale proposals 有真实互补收益



LoG / local peaks 可以做 strong proposal mechanism
	​

最终 detection 仍需要 learned detector / learned score map
	​


In [ ]:
TEST_DIR = DATA_ROOT / "test"

test_zarrs = sorted(
    TEST_DIR.glob("*.zarr")
)

train_by_stem = {
    p.stem: p
    for p in train_zarrs
}

test_by_stem = {
    p.stem: p
    for p in test_zarrs
}

overlap_stems = sorted(
    set(train_by_stem)
    &
    set(test_by_stem)
)

print("Train:", len(train_zarrs))
print("Test:", len(test_zarrs))
print("Stem overlap:", len(overlap_stems))
print()

for stem in overlap_stems:
    print(stem)

In [ ]:
import hashlib

identity_records = []

check_times = [0, 24, 49, 74, 99]

def array_sha256(arr):
    arr = np.ascontiguousarray(arr)
    return hashlib.sha256(
        arr.tobytes()
    ).hexdigest()

for stem in overlap_stems:

    train_path = train_by_stem[stem]
    test_path = test_by_stem[stem]

    train_group = zarr.open_group(
        train_path,
        mode="r",
    )

    test_group = zarr.open_group(
        test_path,
        mode="r",
    )

    train_img = train_group["0"]
    test_img = test_group["0"]

    print(f"\n===== {stem} =====")
    print("Train shape:", train_img.shape)
    print("Test shape: ", test_img.shape)

    print("Train dtype:", train_img.dtype)
    print("Test dtype: ", test_img.dtype)

    print("Train chunks:", train_img.chunks)
    print("Test chunks: ", test_img.chunks)

    for t in check_times:

        train_frame = np.asarray(
            train_img[t]
        )

        test_frame = np.asarray(
            test_img[t]
        )

        exact_equal = np.array_equal(
            train_frame,
            test_frame,
        )

        # Convert before subtraction to avoid uint underflow
        max_abs_diff = np.max(
            np.abs(
                train_frame.astype(np.int64)
                - test_frame.astype(np.int64)
            )
        )

        train_hash = array_sha256(
            train_frame
        )

        test_hash = array_sha256(
            test_frame
        )

        identity_records.append({
            "stem": stem,
            "t": t,
            "exact_equal": exact_equal,
            "max_abs_diff": int(max_abs_diff),
            "train_sha256": train_hash,
            "test_sha256": test_hash,
            "hash_equal":
                train_hash == test_hash,
        })

identity_df = pd.DataFrame(
    identity_records
)

identity_df

In [ ]:
identity_summary = (
    identity_df
    .groupby("stem")
    .agg(
        frames_checked=("t", "size"),
        exact_equal_frames=(
            "exact_equal",
            "sum",
        ),
        hash_equal_frames=(
            "hash_equal",
            "sum",
        ),
        max_abs_diff=(
            "max_abs_diff",
            "max",
        ),
    )
)

identity_summary

In [ ]:
def streaming_zarr_sha256(arr):
    """
    Compute SHA-256 over the entire 4D array in deterministic
    t-major order without loading the whole volume into memory.
    """
    h = hashlib.sha256()

    for t in range(arr.shape[0]):

        frame = np.asarray(
            arr[t]
        )

        frame = np.ascontiguousarray(
            frame
        )

        h.update(
            frame.tobytes()
        )

    return h.hexdigest()

In [ ]:
full_identity_records = []

for stem in overlap_stems:

    print(f"\n===== {stem} =====")

    train_group = zarr.open_group(
        train_by_stem[stem],
        mode="r",
    )

    test_group = zarr.open_group(
        test_by_stem[stem],
        mode="r",
    )

    train_img = train_group["0"]
    test_img = test_group["0"]

    train_hash = streaming_zarr_sha256(
        train_img
    )

    test_hash = streaming_zarr_sha256(
        test_img
    )

    hash_equal = (
        train_hash == test_hash
    )

    print(
        "Full-volume hash equal:",
        hash_equal,
    )

    print(
        "Train SHA256:",
        train_hash,
    )

    print(
        "Test  SHA256:",
        test_hash,
    )

    full_identity_records.append({
        "stem": stem,
        "shape_equal":
            train_img.shape
            == test_img.shape,

        "dtype_equal":
            train_img.dtype
            == test_img.dtype,

        "full_hash_equal":
            hash_equal,

        "train_sha256":
            train_hash,

        "test_sha256":
            test_hash,
    })

full_identity_df = pd.DataFrame(
    full_identity_records
)

full_identity_df[
    [
        "stem",
        "shape_equal",
        "dtype_equal",
        "full_hash_equal",
    ]
]